
# 01 · Procesamiento de Datos — Arquitectura Medallion
**TFM: Análisis y Predicción de la Extorsión en Colombia**

Este script implementa el pipeline ETL completo siguiendo la arquitectura Medallion:
- **Bronze**: Ingesta desde APIs (Socrata / datos.gov.co) y archivos oficiales
- **Silver**: Limpieza, estandarización y validación individual por fuente
- **Gold**: Consolidación y filtrado del dataset analítico para modelado

**Output principal**: `datos/gold/consolidado_modelado.csv` (16.869 registros, 2020–2025)


## 1. Configuración Inicial


In [1]:
import pandas as pd
import numpy as np
import requests
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

# ── Rutas (arquitectura Medallion) ──────────────────────────────────────────
SILVER_PATH = 'datos/silver'   # Datasets limpios individuales
GOLD_PATH   = 'datos/gold'     # Dataset consolidado listo para análisis

os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH,   exist_ok=True)

print(f"[OK] Directorios listos")
print(f"     SILVER : {SILVER_PATH}/")
print(f"     GOLD   : {GOLD_PATH}/")

[OK] Directorios listos
     SILVER : datos/silver/
     GOLD   : datos/gold/



## 2. Funciones Auxiliares


In [2]:
def fetch_api_data(api_url: str, dataset_name: str, batch_size: int = 50000) -> pd.DataFrame:
    """
    Descarga datos desde una API Socrata con paginación automática.
    Maneja datasets grandes (>1000 registros por defecto de la API).
    """
    print(f"Descargando '{dataset_name}'...")
    all_data, offset = [], 0
    try:
        while True:
            url = f"{api_url}?$limit={batch_size}&$offset={offset}&$order=:id"
            response = requests.get(url, timeout=120)
            response.raise_for_status()
            batch = response.json()
            if not batch or not isinstance(batch, list):
                break
            all_data.extend(batch)
            print(f"  ... {len(all_data):,} registros")
            if len(batch) < batch_size:
                break
            offset += batch_size
        if not all_data:
            print(f"  [AVISO] Sin datos para {dataset_name}")
            return pd.DataFrame()
        df = pd.DataFrame(all_data)
        print(f"  [OK] {df.shape[0]:,} filas x {df.shape[1]} columnas")
        return df
    except Exception as e:
        print(f"  [ERROR] {e}")
        return pd.DataFrame()


def parse_date_column(series: pd.Series) -> pd.Series:
    """
    Parsea fechas con formato ISO (APIs Socrata) o formatos comunes.
    Intenta múltiples formatos de forma ordenada.
    """
    for fmt in ('%Y-%m-%dT%H:%M:%S.%f', '%Y-%m-%d', '%d/%m/%Y'):
        result = pd.to_datetime(series, format=fmt, errors='coerce')
        if result.isna().sum() < len(result) * 0.5:
            return result
    return pd.to_datetime(series, infer_datetime_format=True, errors='coerce')


def extract_year_month(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """Agrega columnas `ano` y `mes` a partir de una columna de fecha."""
    df = df.copy()
    df[date_col] = parse_date_column(df[date_col])
    df['ano'] = df[date_col].dt.year
    df['mes'] = df[date_col].dt.month
    return df


def standardize_codes(df: pd.DataFrame, cod_dpto_col: str, cod_mun_col: str = None) -> pd.DataFrame:
    """
    Estandariza códigos DANE:
    - cod_dpto: string 2 dígitos con padding (ej: '05')
    - cod_mun : string 5 dígitos con padding (ej: '05001')
    """
    df = df.copy()
    df['cod_dpto'] = df[cod_dpto_col].astype(str).str.strip().str.zfill(2)
    if cod_mun_col and cod_mun_col in df.columns:
        df['cod_mun'] = df[cod_mun_col].astype(str).str.strip().str.zfill(5)
    return df


def export_silver(df: pd.DataFrame, name: str):
    """Exporta dataset SILVER (limpio, sin agregar)."""
    path = f"{SILVER_PATH}/{name}_silver.csv"
    df.to_csv(path, index=False)
    print(f"  [SILVER] {path}  ({len(df):,} registros)")


def export_gold(df: pd.DataFrame, name: str):
    """Exporta dataset GOLD individual (agregado mensual)."""
    path = f"{GOLD_PATH}/{name}_gold.csv"
    df.to_csv(path, index=False)
    print(f"  [GOLD]   {path}  ({len(df):,} registros)")


print("[OK] Funciones auxiliares definidas")

[OK] Funciones auxiliares definidas



## 3. Capa Bronze → Silver

Cada fuente se descarga en su estado original (Bronze) y luego se limpia
individualmente (Silver). Las transformaciones aplicadas son:
- Nombres de columnas estandarizados (minúsculas, sin tildes)
- Códigos DANE normalizados según Divipola
- Fechas parseadas a formato datetime; extracción de `ano` y `mes`
- Valores numéricos validados y nulos imputados con 0 (ausencia = no ocurrencia)
- Duplicados eliminados



### 3.1 Extorsión (Variable Objetivo)
**Fuente**: Policía Nacional — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/q2ib-t9am.json`


In [3]:
df_extorsion_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/q2ib-t9am.json',
    dataset_name='Extorsion'
)

df_extorsion = df_extorsion_raw.copy()
df_extorsion = standardize_codes(df_extorsion, 'cod_depto', 'cod_muni')
df_extorsion = extract_year_month(df_extorsion, 'fecha_hecho')
df_extorsion['cantidad'] = pd.to_numeric(df_extorsion['cantidad'], errors='coerce').fillna(0).astype(int)
df_extorsion = df_extorsion.rename(columns={'departamento': 'dpto_nombre', 'municipio': 'mun_nombre'})

df_extorsion_silver = df_extorsion[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

export_silver(df_extorsion_silver, 'extorsion')

# Gold: agregado mensual por municipio
df_extorsion_gold = (df_extorsion_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_extorsion=('cantidad', 'sum'))
    .reset_index()
    .astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'mes': int, 'total_extorsion': int})
)
export_gold(df_extorsion_gold, 'extorsion')

Descargando 'Extorsion'...
  ... 50,000 registros
  ... 100,000 registros
  ... 122,457 registros
  [OK] 122,457 filas x 6 columnas
  [SILVER] datos/silver/extorsion_silver.csv  (122,457 registros)
  [GOLD]   datos/gold/extorsion_gold.csv  (38,889 registros)



### 3.2 Homicidio
**Fuente**: Policía Nacional — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/m8fd-ahd9.json`


In [4]:
df_homicidio_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/m8fd-ahd9.json',
    dataset_name='Homicidio'
)

df_homicidio = df_homicidio_raw.copy()
df_homicidio = standardize_codes(df_homicidio, 'cod_depto', 'cod_muni')
df_homicidio = extract_year_month(df_homicidio, 'fecha_hecho')
df_homicidio['cantidad'] = pd.to_numeric(df_homicidio['cantidad'], errors='coerce').fillna(0).astype(int)
df_homicidio = df_homicidio.rename(columns={'departamento': 'dpto_nombre', 'municipio': 'mun_nombre'})

df_homicidio_silver = df_homicidio[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

export_silver(df_homicidio_silver, 'homicidio')

df_homicidio_gold = (df_homicidio_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_homicidio=('cantidad', 'sum'))
    .reset_index()
    .astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'mes': int, 'total_homicidio': int})
)
export_gold(df_homicidio_gold, 'homicidio')

Descargando 'Homicidio'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 335,581 registros
  [OK] 335,581 filas x 11 columnas
  [SILVER] datos/silver/homicidio_silver.csv  (335,581 registros)
  [GOLD]   datos/gold/homicidio_gold.csv  (89,456 registros)



### 3.3 Hurto
**Fuente**: Policía Nacional — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/4rxi-8m8d.json`


In [5]:
df_hurto_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/4rxi-8m8d.json',
    dataset_name='Hurto'
)

df_hurto = df_hurto_raw.copy()
df_hurto = standardize_codes(df_hurto, 'cod_depto', 'cod_muni')
df_hurto = extract_year_month(df_hurto, 'fecha_hecho')
df_hurto['cantidad'] = pd.to_numeric(df_hurto['cantidad'], errors='coerce').fillna(0).astype(int)
df_hurto = df_hurto.rename(columns={'departamento': 'dpto_nombre', 'municipio': 'mun_nombre'})

df_hurto_silver = df_hurto[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

export_silver(df_hurto_silver, 'hurto')

df_hurto_gold = (df_hurto_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_hurto=('cantidad', 'sum'))
    .reset_index()
    .astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'mes': int, 'total_hurto': int})
)
export_gold(df_hurto_gold, 'hurto')

Descargando 'Hurto'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 450,000 registros
  ... 500,000 registros
  ... 550,000 registros
  ... 600,000 registros
  ... 632,758 registros
  [OK] 632,758 filas x 6 columnas
  [SILVER] datos/silver/hurto_silver.csv  (632,758 registros)
  [GOLD]   datos/gold/hurto_gold.csv  (111,316 registros)



### 3.4 Terrorismo
**Fuente**: Policía Nacional — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/yi5j-5fe9.json`


In [6]:
df_terrorismo_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/yi5j-5fe9.json',
    dataset_name='Terrorismo'
)

df_terrorismo = df_terrorismo_raw.copy()
df_terrorismo = standardize_codes(df_terrorismo, 'cod_depto', 'cod_muni')
df_terrorismo = extract_year_month(df_terrorismo, 'fecha_hecho')
df_terrorismo['cantidad'] = pd.to_numeric(df_terrorismo['cantidad'], errors='coerce').fillna(0).astype(int)
df_terrorismo = df_terrorismo.rename(columns={'departamento': 'dpto_nombre', 'municipio': 'mun_nombre'})

df_terrorismo_silver = df_terrorismo[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

export_silver(df_terrorismo_silver, 'terrorismo')

df_terrorismo_gold = (df_terrorismo_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_terrorismo=('cantidad', 'sum'))
    .reset_index()
    .astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'mes': int, 'total_terrorismo': int})
)
export_gold(df_terrorismo_gold, 'terrorismo')

Descargando 'Terrorismo'...
  ... 14,897 registros
  [OK] 14,897 filas x 7 columnas
  [SILVER] datos/silver/terrorismo_silver.csv  (14,897 registros)
  [GOLD]   datos/gold/terrorismo_gold.csv  (9,552 registros)



### 3.5 Estupefacientes
**Fuente**: Policía Nacional — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/kk69-w2jj.json`
**Nota**: Las incautaciones se agregan como conteo de eventos (`incautaciones_total`)
y como volumen en kg (`cantidad_total_kg`).


In [7]:
df_estupef_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/kk69-w2jj.json',
    dataset_name='Estupefacientes'
)

df_estupefacientes = df_estupef_raw.copy()
df_estupefacientes = df_estupefacientes.rename(columns={
    'departamento': 'dpto_nombre', 'municipio': 'mun_nombre',
    'codigo_dane': 'cod_mun', 'clase_bien': 'tipo_sustancia',
    'fecha_hecho': 'fecha', 'cantidad': 'cantidad'
})
df_estupefacientes['fecha']    = pd.to_datetime(df_estupefacientes['fecha'], format='%d/%m/%Y', errors='coerce')
df_estupefacientes['ano']      = df_estupefacientes['fecha'].dt.year
df_estupefacientes['mes']      = df_estupefacientes['fecha'].dt.month
df_estupefacientes['cod_mun']  = df_estupefacientes['cod_mun'].astype(str).str.zfill(8).str[:5]
df_estupefacientes['cod_dpto'] = df_estupefacientes['cod_mun'].str[:2]
df_estupefacientes['cantidad'] = pd.to_numeric(df_estupefacientes['cantidad'], errors='coerce').fillna(0)
df_estupefacientes['tipo_sustancia'] = (df_estupefacientes['tipo_sustancia']
    .str.upper().str.strip().str.replace(' ', '_'))

df_estupef_silver = df_estupefacientes[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha', 'ano', 'mes', 'tipo_sustancia', 'cantidad'
]].copy()

export_silver(df_estupef_silver, 'estupefacientes')

df_estupef_gold = (df_estupef_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(incautaciones_total=('cantidad', 'count'), cantidad_total_kg=('cantidad', 'sum'))
    .reset_index()
    .astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'mes': int, 'incautaciones_total': int})
)
export_gold(df_estupef_gold, 'estupefacientes')

Descargando 'Estupefacientes'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 450,000 registros
  ... 500,000 registros
  ... 550,000 registros
  ... 600,000 registros
  ... 650,000 registros
  ... 700,000 registros
  ... 750,000 registros
  ... 800,000 registros
  ... 850,000 registros
  ... 900,000 registros
  ... 950,000 registros
  ... 1,000,000 registros
  ... 1,050,000 registros
  ... 1,100,000 registros
  ... 1,150,000 registros
  ... 1,200,000 registros
  ... 1,250,000 registros
  ... 1,300,000 registros
  ... 1,350,000 registros
  ... 1,400,000 registros
  ... 1,450,000 registros
  ... 1,500,000 registros
  ... 1,550,000 registros
  ... 1,600,000 registros
  ... 1,650,000 registros
  ... 1,700,000 registros
  ... 1,750,000 registros
  ... 1,800,000 registros
  ... 1,850,000 registros
  ... 1,900,000 registros
  ... 1,928,185 re


### 3.6 Coca (Cultivos Ilícitos)
**Fuente**: UNODC / SIMCI — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/acs4-3wgp.json`
**Nota**: Dataset de granularidad anual (no mensual). Contiene hectáreas de coca por municipio.


In [8]:
df_coca_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/acs4-3wgp.json',
    dataset_name='Coca'
)

df_coca = df_coca_raw.copy().rename(columns={
    'coddepto': 'cod_dpto', 'codmpio': 'cod_mun',
    'departamento': 'dpto_nombre', 'municipio': 'mun_nombre'
})

year_cols = [c for c in df_coca.columns if c.startswith('_') and c[1:].isdigit()]
df_coca_long = df_coca.melt(
    id_vars=['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre'],
    value_vars=year_cols, var_name='ano', value_name='hectareas'
)
df_coca_long['ano'] = df_coca_long['ano'].str.replace('_', '').astype(int)
df_coca_long['hectareas'] = (df_coca_long['hectareas'].astype(str)
    .str.replace('- 0', '0').str.replace('-', '').str.strip())
df_coca_long['hectareas'] = pd.to_numeric(df_coca_long['hectareas'], errors='coerce').fillna(0)
df_coca_long['cod_dpto'] = df_coca_long['cod_dpto'].astype(str).str.zfill(2)
df_coca_long['cod_mun']  = df_coca_long['cod_mun'].astype(str).str.zfill(5)

df_coca_silver = df_coca_long[['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'hectareas']].copy()
df_coca_silver = df_coca_silver.astype({'cod_dpto': str, 'cod_mun': str, 'ano': int, 'hectareas': float})
export_silver(df_coca_silver, 'coca')
export_gold(df_coca_silver.rename(columns={'hectareas': 'hectareas_coca'}), 'coca')

Descargando 'Coca'...
  ... 319 registros
  [OK] 319 filas x 27 columnas
  [SILVER] datos/silver/coca_silver.csv  (7,337 registros)
  [GOLD]   datos/gold/coca_gold.csv  (7,337 registros)



### 3.7 Población Municipal
**Fuente**: DANE — Proyecciones de Población 2020–2035 (post-COVID)
**Nota**: Datos anuales. Se expanden a mensual en la etapa Gold para el merge.


In [9]:
excel_url = ("https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion"
             "/Municipal/DCD-area-proypoblacion-Mun-2020-2035-ActPostCOVID-19.xlsx")
print("Descargando 'Poblacion' (DANE Excel)...")
df_poblacion_raw = pd.read_excel(excel_url, sheet_name="Hoja1", header=8)
print(f"  [OK] {df_poblacion_raw.shape[0]:,} filas")

df_poblacion = df_poblacion_raw.rename(columns={
    'DP': 'cod_dpto', 'DPNOM': 'dpto_nombre', 'MPIO': 'cod_mun',
    'DPMP': 'mun_nombre', 'AÑO': 'ano', 'Población': 'poblacion',
    'ÁREA GEOGRÁFICA': 'area_geografica'
})
df_poblacion = df_poblacion[df_poblacion['ano'].notna() & df_poblacion['cod_mun'].notna()]

area_map = {'Cabecera Municipal': 'Urbana', 'Centros Poblados y Rural Disperso': 'Rural', 'Total': 'Total'}
df_poblacion = df_poblacion[df_poblacion['area_geografica'].isin(area_map)].copy()
df_poblacion['tipo_area'] = df_poblacion['area_geografica'].map(area_map)
df_poblacion['cod_dpto']  = df_poblacion['cod_dpto'].astype(str).str.zfill(2)
df_poblacion['cod_mun']   = df_poblacion['cod_mun'].astype(str).str.split('.').str[0].str.zfill(5)
df_poblacion['ano']       = df_poblacion['ano'].astype(int)
df_poblacion['poblacion'] = pd.to_numeric(df_poblacion['poblacion'], errors='coerce').astype('Int64')

df_poblacion_silver = df_poblacion[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'tipo_area', 'poblacion'
]].copy()
export_silver(df_poblacion_silver, 'poblacion')

# Gold: pivotar Urbana / Rural / Total como columnas
df_poblacion_gold = (df_poblacion_silver
    .pivot_table(index=['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano'],
                 columns='tipo_area', values='poblacion', aggfunc='first')
    .reset_index()
)
df_poblacion_gold.columns.name = None
df_poblacion_gold = df_poblacion_gold.rename(columns={
    'Rural': 'poblacion_rural', 'Total': 'poblacion_total', 'Urbana': 'poblacion_urbana'
}).astype({'ano': int})
export_gold(df_poblacion_gold, 'poblacion')

Descargando 'Poblacion' (DANE Excel)...
  [OK] 53,859 filas
  [SILVER] datos/silver/poblacion_silver.csv  (53,856 registros)
  [GOLD]   datos/gold/poblacion_gold.csv  (17,952 registros)



### 3.8 PIB Departamental
**Fuente**: DANE — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/kgyi-qc7j.json`
**Nota**: Granularidad departamental y anual. Se calcula PIB per cápita usando población DANE.


In [10]:
df_pib_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/kgyi-qc7j.json',
    dataset_name='PIB'
)

df_pib = df_pib_raw.copy().rename(columns={
    'a_o': 'ano', 'c_digo_departamento_divipola': 'cod_dpto',
    'departamento': 'dpto_nombre', 'valor_miles_de_millones_de': 'valor_agregado'
})
df_pib['cod_dpto']       = df_pib['cod_dpto'].astype(str).str.zfill(2)
df_pib['ano']            = pd.to_numeric(df_pib['ano'], errors='coerce').astype('Int64')
df_pib['valor_agregado'] = pd.to_numeric(df_pib['valor_agregado'], errors='coerce')
export_silver(df_pib.copy(), 'pib')

df_pib_gold = (df_pib.groupby(['cod_dpto', 'dpto_nombre', 'ano'])
    .agg(pib_total=('valor_agregado', 'sum')).reset_index())

# PIB per cápita: suma población departamental desde el gold de población
df_pob_dpto = df_poblacion_gold.groupby(['cod_dpto', 'ano'], as_index=False).agg({'poblacion_total': 'sum'})
df_pib_gold = df_pib_gold.merge(df_pob_dpto, on=['cod_dpto', 'ano'], how='left')
df_pib_gold['pib_per_capita'] = (df_pib_gold['pib_total'] * 1e9 / df_pib_gold['poblacion_total']).round(2)
df_pib_gold = df_pib_gold.astype({'cod_dpto': str, 'ano': int})
export_gold(df_pib_gold, 'pib_departamental')

Descargando 'PIB'...
  ... 16,302 registros
  [OK] 16,302 filas x 7 columnas
  [SILVER] datos/silver/pib_silver.csv  (16,302 registros)
  [GOLD]   datos/gold/pib_departamental_gold.csv  (627 registros)



### 3.9 Atentados Terroristas
**Fuente**: OCHA Colombia — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/yfd7-8c9d.json`


In [11]:
df_atentados_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/yfd7-8c9d.json',
    dataset_name='Atentados'
)

df_atentados = df_atentados_raw.copy().rename(columns={
    'a_o': 'ano', 'c_digo_dane_de_municipio': 'cod_mun',
    'departamento': 'dpto_nombre', 'municipio': 'mun_nombre',
    'total_de_v_ctimas_del_caso': 'total_victimas',
    'abandono_o_despojo_forzado': 'abandono_despojo',
    'amenaza_o_intimidaci_n': 'amenaza_intimidacion',
    'ataque_contra_misi_n_m_dica': 'ataque_mision_medica',
    'confinamiento_o_restricci': 'confinamiento',
    'extorsi_n': 'extorsion_asociada'
})
df_atentados['cod_mun']  = df_atentados['cod_mun'].astype(str).str.zfill(5)
df_atentados['cod_dpto'] = df_atentados['cod_mun'].str[:2]
df_atentados['ano']      = pd.to_numeric(df_atentados['ano'], errors='coerce').astype('Int64')
df_atentados['mes']      = pd.to_numeric(df_atentados.get('mes', pd.Series()), errors='coerce').astype('Int64')

num_cols = ['total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles',
            'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica',
            'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada', 'pillaje', 'tortura']
for col in num_cols:
    if col in df_atentados.columns:
        df_atentados[col] = pd.to_numeric(df_atentados[col], errors='coerce').fillna(0).astype(int)

silver_cols = ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'] + \
              [c for c in num_cols if c in df_atentados.columns]
df_atentados_silver = df_atentados[silver_cols].copy()
export_silver(df_atentados_silver, 'atentados')

agg_cols = {c: 'sum' for c in num_cols if c in df_atentados_silver.columns}
df_atentados_gold = (df_atentados_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(agg_cols).reset_index()
)
export_gold(df_atentados_gold, 'atentados')

Descargando 'Atentados'...
  ... 282 registros
  [OK] 282 filas x 27 columnas
  [SILVER] datos/silver/atentados_silver.csv  (282 registros)
  [GOLD]   datos/gold/atentados_gold.csv  (268 registros)



### 3.10 Cobertura Móvil
**Fuente**: MinTIC — datos.gov.co
**URL API**: `https://www.datos.gov.co/resource/9mey-c8s8.json`
**Nota**: Variables binarias (S/N) convertidas a 1/0.


In [12]:
df_cobertura_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/9mey-c8s8.json',
    dataset_name='Cobertura Movil'
)

df_cobertura = df_cobertura_raw.copy().rename(columns={
    'a_o': 'ano', 'cod_departamento': 'cod_dpto', 'departamento': 'dpto_nombre',
    'cod_municipio': 'cod_mun', 'municipio': 'mun_nombre', 'cobertuta_4g': 'cobertura_4g'
})
df_cobertura['cod_dpto'] = df_cobertura['cod_dpto'].astype(str).str.zfill(2)
df_cobertura['cod_mun']  = df_cobertura['cod_mun'].astype(str).str.zfill(5)
df_cobertura['ano']      = pd.to_numeric(df_cobertura['ano'], errors='coerce').astype('Int64')

cob_cols = [c for c in ['cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']
            if c in df_cobertura.columns]
for col in cob_cols:
    df_cobertura[col] = df_cobertura[col].map({'S': 1, 'N': 0}).fillna(0).astype(int)

export_silver(df_cobertura.copy(), 'cobertura_movil')

df_cobertura_gold = (df_cobertura
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano'])
    .agg({c: 'max' for c in cob_cols}).reset_index()
)
export_gold(df_cobertura_gold, 'cobertura_movil')

Descargando 'Cobertura Movil'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 407,281 registros
  [OK] 407,281 filas x 16 columnas
  [SILVER] datos/silver/cobertura_movil_silver.csv  (407,281 registros)
  [GOLD]   datos/gold/cobertura_movil_gold.csv  (10,095 registros)



## 4. Capa Gold — Consolidación Analítica

Se construye el **Tablón Analítico** (Analytical Base Table) integrando los datasets
Silver mediante llaves compuestas: `cod_dpto` + `cod_mun` + `ano` + `mes`.

**Datasets mensuales incluidos** (merge directo):
- Extorsión, Homicidio, Hurto, Terrorismo, Estupefacientes, Población (expandida a mensual)

**Datasets anuales excluidos del consolidado** (causan duplicaciones en el merge mensual):
- Coca, PIB, Cobertura móvil, Atentados
Estos están disponibles individualmente en `datos/gold/` para análisis separados.


In [13]:
# Expandir Población de anual a mensual (cross-join con meses 1–12)
df_poblacion_gold['_key'] = 1
meses_df = pd.DataFrame({'mes': range(1, 13), '_key': 1})
df_poblacion_mensual = df_poblacion_gold.merge(meses_df, on='_key').drop('_key', axis=1)

print(f"Población expandida: {len(df_poblacion_mensual):,} registros (anual x 12 meses)")

# Base del consolidado: extorsión
merge_keys = ['cod_dpto', 'cod_mun', 'ano', 'mes']
df_consolidado = df_extorsion_gold[['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
                                     'ano', 'mes', 'total_extorsion']].copy()
print(f"\nBase (extorsion): {len(df_consolidado):,} registros")

# Merge con datasets mensuales
datasets_mensuales = [
    ('homicidio',       df_homicidio_gold,   ['total_homicidio']),
    ('hurto',           df_hurto_gold,        ['total_hurto']),
    ('terrorismo',      df_terrorismo_gold,   ['total_terrorismo']),
    ('estupefacientes', df_estupef_gold,      ['incautaciones_total', 'cantidad_total_kg']),
    ('poblacion',       df_poblacion_mensual, ['poblacion_total']),
]

for name, df, cols in datasets_mensuales:
    n_antes = len(df_consolidado)
    df_consolidado = df_consolidado.merge(df[merge_keys + cols], on=merge_keys, how='left')
    n_despues = len(df_consolidado)
    flag = '[OK]' if n_antes == n_despues else '[WARN] DUPLICACION!'
    print(f"  + {name:<16} {n_despues:>8,} filas  {flag}")

print(f"\nConsolidado completo: {len(df_consolidado):,} filas x {len(df_consolidado.columns)} columnas")
print(f"Periodo: {df_consolidado['ano'].min()} – {df_consolidado['ano'].max()}")

# Guardar consolidado completo
consolidado_path = f"{GOLD_PATH}/consolidado_analitico.csv"
df_consolidado.to_csv(consolidado_path, index=False)
print(f"[GOLD] {consolidado_path}")

Población expandida: 215,424 registros (anual x 12 meses)

Base (extorsion): 38,889 registros
  + homicidio          38,889 filas  [OK]
  + hurto              38,889 filas  [OK]
  + terrorismo         38,889 filas  [OK]
  + estupefacientes    38,894 filas  [WARN] DUPLICACION!
  + poblacion          38,894 filas  [OK]

Consolidado completo: 38,894 filas x 13 columnas
Periodo: 2003 – 2026
[GOLD] datos/gold/consolidado_analitico.csv



## 5. Dataset de Modelado (Filtrado)

Se filtra el consolidado para construir el dataset final de modelado:
- **Período**: 2020–2025 (6 años con datos de población DANE disponibles)
- **Criterio de inclusión**: Solo municipios con al menos 1 caso de extorsión
  registrado en el período. Esto evita ceros estructurales que distorsionarían
  las métricas del modelo y asegura que el target siempre sea positivo.

**Resultado**: 16.869 observaciones municipio-mes en 1.046 municipios activos.


In [14]:
ANO_INICIO, ANO_FIN = 2020, 2025

df_modelado = df_consolidado[
    (df_consolidado['ano'] >= ANO_INICIO) &
    (df_consolidado['ano'] <= ANO_FIN) &
    (df_consolidado['total_extorsion'] > 0)
].copy()

# Rellenar nulos en predictoras con 0 (ausencia de evento = cero)
predictoras = ['total_homicidio', 'total_hurto', 'total_terrorismo',
               'incautaciones_total', 'cantidad_total_kg', 'poblacion_total']
df_modelado[predictoras] = df_modelado[predictoras].fillna(0)

modelado_path = f"{GOLD_PATH}/consolidado_modelado.csv"
df_modelado.to_csv(modelado_path, index=False)

print("=" * 65)
print("DATASET DE MODELADO EXPORTADO")
print("=" * 65)
print(f"Archivo    : {modelado_path}")
print(f"Registros  : {len(df_modelado):,}")
print(f"Periodo    : {df_modelado['ano'].min()} – {df_modelado['ano'].max()}")
print(f"Dptos      : {df_modelado['cod_dpto'].nunique()}")
print(f"Municipios : {df_modelado['cod_mun'].nunique()}")
print(f"Columnas   : {list(df_modelado.columns)}")
print()
print("Registros por año:")
print(df_modelado.groupby('ano').size().to_string())

DATASET DE MODELADO EXPORTADO
Archivo    : datos/gold/consolidado_modelado.csv
Registros  : 16,869
Periodo    : 2020 – 2025
Dptos      : 33
Municipios : 1046
Columnas   : ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_extorsion', 'total_homicidio', 'total_hurto', 'total_terrorismo', 'incautaciones_total', 'cantidad_total_kg', 'poblacion_total']

Registros por año:
ano
2020    2475
2021    2409
2022    2729
2023    2874
2024    3307
2025    3075



## 6. Resumen de Archivos Generados


In [15]:
print("=" * 65)
print("RESUMEN DE ARCHIVOS GENERADOS")
print("=" * 65)
for folder in [SILVER_PATH, GOLD_PATH]:
    print(f"\n{folder}/")
    for f in sorted(os.listdir(folder)):
        size_mb = os.path.getsize(os.path.join(folder, f)) / 1024 / 1024
        print(f"  {f:<40} {size_mb:.2f} MB")

RESUMEN DE ARCHIVOS GENERADOS

datos/silver/
  atentados_silver.csv                     0.02 MB
  cobertura_movil_silver.csv               40.81 MB
  coca_silver.csv                          0.29 MB
  estupefacientes_silver.csv               117.66 MB
  extorsion_silver.csv                     5.89 MB
  homicidio_silver.csv                     16.30 MB
  hurto_silver.csv                         30.23 MB
  pib_silver.csv                           2.37 MB
  poblacion_silver.csv                     2.40 MB
  terrorismo_silver.csv                    0.71 MB

datos/gold/
  atentados_gold.csv                       0.02 MB
  cobertura_movil_gold.csv                 0.43 MB
  coca_gold.csv                            0.29 MB
  consolidado_analitico.csv                2.16 MB
  consolidado_modelado.csv                 1.10 MB
  estupefacientes_gold.csv                 4.62 MB
  extorsion_gold.csv                       1.44 MB
  homicidio_gold.csv                       3.32 MB
  hurto_gold.csv   

In [18]:
import pandas as pd                                                                                                                      
import os                                                                                                                                                                                                                                                                            
silver_path = 'datos/silver'                                                                                                   
print(f'{'Dataset':<30} {'Año Min':>8} {'Año Max':>8} {'Total':>8}')
import os    
        
silver_path = 'datos/silver'
print(f'{'Dataset':<30} {'Año Min':>8} {'Año Max':>8} {'Total':>8}')
print('-'*56)                              
for f in sorted(os.listdir(silver_path)):                                  
   if not f.endswith('.csv'):
       continue                                                        
   df = pd.read_csv(f'{silver_path}/{f}', low_memory=False)
   col = next((c for c in ['ano','year','anio'] if c in df.columns), None)
   if col:
       print(f'{f:<30} {int(df[col].min()):>8} {int(df[col].max()):>8} {len(df):>8,}')
print()         
print('=== DETALLE 2024-2026 por dataset ===')
for f in sorted(os.listdir(silver_path)):                                  
   if not f.endswith('.csv'):       
       continue 
   df = pd.read_csv(f'{silver_path}/{f}', low_memory=False)
   col = next((c for c in ['ano','year','anio'] if c in df.columns), None)
   if col and df[col].max() >= 2024:
       recent = df[df[col]>=2024].groupby(col).size()
       print(f'{f}: {recent.to_dict()}')

Dataset                         Año Min  Año Max    Total
Dataset                         Año Min  Año Max    Total
--------------------------------------------------------
atentados_silver.csv               1944     2024      282
cobertura_movil_silver.csv         2015     2023  407,281
coca_silver.csv                    2001     2023    7,337
estupefacientes_silver.csv         2010     2025 1,928,185
extorsion_silver.csv               2003     2026  122,457
homicidio_silver.csv               2003     2026  335,581
hurto_silver.csv                   2003     2026  632,758
pib_silver.csv                     2005     2023   16,302
poblacion_silver.csv               2020     2035   53,856
terrorismo_silver.csv              2003     2026   14,897

=== DETALLE 2024-2026 por dataset ===
atentados_silver.csv: {2024: 3}
estupefacientes_silver.csv: {2024: 60801, 2025: 56554}
extorsion_silver.csv: {2024: 13802, 2025: 12180, 2026: 1205}
homicidio_silver.csv: {2024: 13414, 2025: 13659, 2026: 2215